# Qwen3 Forced Aligner Plugin

> Plugin implementation for Qwen3 word-level forced alignment

In [ ]:
#| default_exp plugin

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import logging
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Union
from uuid import uuid4

import torch

try:
    from qwen_asr import Qwen3ForcedAligner
    QWEN3_AVAILABLE = True
except ImportError:
    QWEN3_AVAILABLE = False

from cjm_transcription_plugin_system.forced_alignment_interface import ForcedAlignmentPlugin
from cjm_transcription_plugin_system.forced_alignment_core import ForcedAlignItem, ForcedAlignResult
from cjm_transcription_plugin_system.forced_alignment_storage import ForcedAlignmentStorage
from cjm_plugin_system.utils.hashing import hash_file, hash_bytes
from cjm_plugin_system.core.interface import RELOAD_TRIGGER
from cjm_plugin_system.core.errors import (
    PluginInputError, PluginResourceError, ResourceShortfall,
)
from cjm_plugin_system.utils.validation import (
    dict_to_config, config_to_dict, dataclass_to_jsonschema,
    SCHEMA_TITLE, SCHEMA_DESC, SCHEMA_ENUM
)

from cjm_transcription_plugin_qwen3_forced_aligner.meta import get_plugin_metadata

/home/innom-dt/miniforge3/envs/cjm-transcription-plugin-qwen3-forced-aligner/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

In [ ]:
#| export
@dataclass
class Qwen3ForcedAlignerConfig:
    """Configuration for the Qwen3 Forced Aligner plugin."""

    model_id: str = field(
        default="Qwen/Qwen3-ForcedAligner-0.6B",
        metadata={
            SCHEMA_TITLE: "Model ID",
            RELOAD_TRIGGER: "model",  # CR-4: change triggers model reload
            SCHEMA_DESC: "HuggingFace model identifier",
        }
    )

    device: str = field(
        default="auto",
        metadata={
            SCHEMA_TITLE: "Device",
            RELOAD_TRIGGER: "model",  # CR-4: change triggers model reload
            SCHEMA_DESC: "Device for model inference ('auto', 'cuda:0', 'cpu')",
        }
    )

    dtype: str = field(
        default="bfloat16",
        metadata={
            SCHEMA_TITLE: "Data Type",
            RELOAD_TRIGGER: "model",  # CR-4: change triggers model reload
            SCHEMA_DESC: "Model precision",
            SCHEMA_ENUM: ["bfloat16", "float16", "float32"],
        }
    )

    attn_implementation: str = field(
        default="sdpa",
        metadata={
            SCHEMA_TITLE: "Attention Implementation",
            RELOAD_TRIGGER: "model",  # CR-4: change triggers model reload
            SCHEMA_DESC: "Attention backend ('sdpa' for PyTorch native, 'flash_attention_2' if flash-attn installed)",
            SCHEMA_ENUM: ["sdpa", "flash_attention_2", "eager"],
        }
    )

    language: str = field(
        default="English",
        metadata={
            SCHEMA_TITLE: "Language",
            SCHEMA_DESC: "Language for alignment",
        }
    )

## Plugin Implementation

In [ ]:
#| export
class Qwen3ForcedAlignerPlugin(ForcedAlignmentPlugin):
    """Qwen3 Forced Alignment plugin for word-level audio-text alignment."""

    config_class = Qwen3ForcedAlignerConfig

    def __init__(self):
        """Initialize the plugin with default state."""
        self.logger = logging.getLogger(f"{__name__}.{type(self).__name__}")
        self._config: Qwen3ForcedAlignerConfig = None
        self._model = None
        self._loaded_model_id: Optional[str] = None
        self._loaded_device: Optional[str] = None
        self._loaded_dtype: Optional[str] = None
        self._loaded_attn: Optional[str] = None
        self._storage: Optional[ForcedAlignmentStorage] = None

    @property
    def name(self) -> str:  # Plugin name identifier
        return "cjm-transcription-plugin-qwen3-forced-aligner"

    @property
    def version(self) -> str:  # Plugin version string
        return "0.0.1"

    @property
    def supported_formats(self) -> List[str]:  # Supported audio file formats
        return ["wav", "mp3", "flac", "ogg", "m4a"]

    def is_available(self) -> bool:  # True if qwen_asr is importable
        """Check if the Qwen3 forced aligner is available."""
        return QWEN3_AVAILABLE

    def get_config_schema(self) -> Dict[str, Any]:  # JSON Schema for configuration
        """Return JSON Schema for UI generation."""
        return dataclass_to_jsonschema(Qwen3ForcedAlignerConfig)

    def get_current_config(self) -> Dict[str, Any]:  # Current configuration as dictionary
        """Return current configuration state."""
        if not self._config:
            return {}
        return config_to_dict(self._config)

    def _apply_config(
        self,
        config: Optional[Any] = None  # Configuration dataclass, dict, or None
    ) -> None:
        """CR-4: apply config values only. Called by initialize (first-time) and the
        substrate's reconfigure delta path. Model release on a model_id/device/dtype/
        attn_implementation change is handled declaratively via RELOAD_TRIGGER ->
        _release_model (device/dtype are resolved lazily in _load_model)."""
        self._config = dict_to_config(Qwen3ForcedAlignerConfig, config or {})

    def initialize(
        self,
        config: Optional[Any] = None  # Configuration dataclass, dict, or None
    ) -> None:
        """First-time setup. CR-4: the manual diff-and-reload is replaced by declarative
        RELOAD_TRIGGER metadata; the substrate's reconfigure path fires _release_model
        then re-applies config via _apply_config."""
        self._apply_config(config)

        # Initialize storage (one-time)
        db_path = get_plugin_metadata().get("db_path")
        if db_path:
            self._storage = ForcedAlignmentStorage(db_path)

        self.logger.info(f"Initialized with model={self._config.model_id}, device={self._config.device}")

    def _load_model(self):
        """Load model on first use. Reports progress for download + load."""
        if self._model is not None:
            return

        self.report_progress(0.0, "Loading Qwen3 Forced Aligner model...")

        device = self._config.device
        if device == "auto":
            device = "cuda:0" if torch.cuda.is_available() else "cpu"

        dtype_map = {
            "bfloat16": torch.bfloat16,
            "float16": torch.float16,
            "float32": torch.float32,
        }
        dtype = dtype_map.get(self._config.dtype, torch.bfloat16)

        self.report_progress(0.1, f"Loading model {self._config.model_id}...")
        self.logger.info(f"Loading model {self._config.model_id} on {device} with {self._config.dtype}")

        self._model = Qwen3ForcedAligner.from_pretrained(
            self._config.model_id,
            dtype=dtype,
            device_map=device,
            attn_implementation=self._config.attn_implementation,
        )

        self._loaded_model_id = self._config.model_id
        self._loaded_device = device
        self._loaded_dtype = self._config.dtype
        self._loaded_attn = self._config.attn_implementation

        self.report_progress(0.25, "Model loaded.")
        self.logger.info("Model loaded successfully")

    def _release_model(self):
        """Release model and free GPU memory."""
        if self._model is not None:
            self.logger.info("Unloading model")
            del self._model
            self._model = None
            self._loaded_model_id = None
            self._loaded_device = None
            self._loaded_dtype = None
            self._loaded_attn = None
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    def execute(
        self,
        audio: Union[str, Path],  # Audio data or file path
        text: str,                            # Transcript text to align against
        **kwargs
    ) -> ForcedAlignResult:  # Word-level alignment result
        """Perform forced alignment of text against audio."""
        # Resolve audio path (caller provides a decodable file path)
        if isinstance(audio, (str, Path)):
            audio_path = str(audio)
        else:
            raise PluginInputError(  # SG-47: typed input-validation (multi-inherits ValueError)
                f"Unsupported audio type: {type(audio)}; expected a file path (str or Path)",
                fields_invalid=["audio"],
            )

        # Lazy load model
        self._load_model()

        self.report_progress(0.3, "Hashing input files...")
        audio_hash = hash_file(audio_path)
        text_hash = hash_bytes(text.encode("utf-8"))

        self.report_progress(0.4, "Running forced alignment...")
        self.logger.info(f"Running alignment on {audio_path} ({len(text)} chars)")

        # Run alignment. SG-47 Track B wraps the inference site so CUDA OOM
        # surfaces as PluginResourceError → CR-7 reactive-retry reloads.
        language = kwargs.get("language", self._config.language)
        try:
            results = self._model.align(
                audio=audio_path,
                text=text,
                language=language,
            )
        except torch.cuda.OutOfMemoryError as e:
            free_bytes = torch.cuda.mem_get_info()[0] if torch.cuda.is_available() else 0
            available_mb = free_bytes / (1024 ** 2)
            raise PluginResourceError(
                f"CUDA OOM during Qwen3 forced-alignment (audio_chars={len(text)}): {e}",
                resource_shortfall=ResourceShortfall(
                    resource='gpu_vram_mb',
                    needed=available_mb + 100.0,
                    available=available_mb,
                ),
            ) from e

        self.report_progress(0.8, "Processing results...")

        # Convert qwen_asr ForcedAlignResult to our standardized format
        items = []
        for item in results[0].items:
            items.append(ForcedAlignItem(
                text=item.text,
                start_time=item.start_time,
                end_time=item.end_time,
            ))

        result = ForcedAlignResult(
            items=items,
            metadata={
                "model_id": self._config.model_id,
                "language": language,
                "device": self._loaded_device,
                "dtype": self._config.dtype,
                "word_count": len(items),
            }
        )

        # Store result
        if self._storage:
            job_id = kwargs.get("job_id", str(uuid4()))
            if self._storage.save_with_logging(
                job_id=job_id,
                audio_path=audio_path,
                audio_hash=audio_hash,
                text=text,
                text_hash=text_hash,
                items=[asdict(item) for item in items],
                metadata=result.metadata,
                logger=self.logger,
            ):
                result.metadata["job_id"] = job_id

        self.report_progress(1.0, "Alignment complete.")
        self.logger.info(f"Alignment complete: {len(items)} words")
        return result

    def prefetch(self) -> None:
        """CR-4 (SG-19): eagerly load the model so the first execute() doesn't pay
        the download/load cost. Idempotent via _load_model's None-guard."""
        self._load_model()

    def on_disable(self) -> None:
        """CR-2: release the GPU model when the operator disables the plugin (the
        worker stays alive); lazy reload on the next execute after re-enable."""
        self._release_model()

    def cleanup(self) -> None:
        """Clean up resources."""
        self._release_model()
        self.logger.info("Cleanup completed")

## Testing

In [ ]:
# Test basic plugin properties
plugin = Qwen3ForcedAlignerPlugin()

print(f"Plugin name: {plugin.name}")
print(f"Plugin version: {plugin.version}")
print(f"Supported formats: {plugin.supported_formats}")
print(f"Entry point group: {plugin.entry_point_group}")
print(f"Available: {plugin.is_available()}")
print(f"Config class: {plugin.config_class.__name__}")

Plugin name: cjm-transcription-plugin-qwen3-forced-aligner
Plugin version: 0.0.1
Supported formats: ['wav', 'mp3', 'flac', 'ogg', 'm4a']
Entry point group: transcription.forced_alignment_plugins
Available: True
Config class: Qwen3ForcedAlignerConfig


In [ ]:
# Test configuration
from dataclasses import fields

print("Config fields:")
for f in fields(Qwen3ForcedAlignerConfig):
    print(f"  {f.name}: default={f.default}, title={f.metadata.get(SCHEMA_TITLE, '')}")

Config fields:
  model_id: default=Qwen/Qwen3-ForcedAligner-0.6B, title=Model ID
  device: default=auto, title=Device
  dtype: default=bfloat16, title=Data Type
  attn_implementation: default=sdpa, title=Attention Implementation
  language: default=English, title=Language


In [ ]:
# Test JSON Schema generation
import json

plugin = Qwen3ForcedAlignerPlugin()

schema = plugin.get_config_schema()
print(f"Schema name: {schema['name']}")
print(f"Properties: {len(schema['properties'])}")
print(json.dumps(schema['properties'], indent=2))

Schema name: Qwen3ForcedAlignerConfig
Properties: 5
{
  "model_id": {
    "type": "string",
    "title": "Model ID",
    "description": "HuggingFace model identifier",
    "enum": [
      "Qwen/Qwen3-ForcedAligner-0.6B"
    ],
    "default": "Qwen/Qwen3-ForcedAligner-0.6B"
  },
  "device": {
    "type": "string",
    "title": "Device",
    "description": "Device for model inference ('auto', 'cuda:0', 'cpu')",
    "default": "auto"
  },
  "dtype": {
    "type": "string",
    "title": "Data Type",
    "description": "Model precision",
    "enum": [
      "bfloat16",
      "float16",
      "float32"
    ],
    "default": "bfloat16"
  },
  "attn_implementation": {
    "type": "string",
    "title": "Attention Implementation",
    "description": "Attention backend ('sdpa' for PyTorch native, 'flash_attention_2' if flash-attn installed)",
    "enum": [
      "sdpa",
      "flash_attention_2",
      "eager"
    ],
    "default": "sdpa"
  },
  "language": {
    "type": "string",
    "title": "

In [ ]:
# Test initialization
import logging
logging.basicConfig(level=logging.INFO)

plugin.initialize({"language": "English", "device": "auto"})
print(f"Current config: {plugin.get_current_config()}")

# Model should NOT be loaded yet (lazy loading)
assert plugin._model is None
print("Model not loaded (lazy): OK")

Initialized with model=Qwen/Qwen3-ForcedAligner-0.6B, device=auto


Current config: {'model_id': 'Qwen/Qwen3-ForcedAligner-0.6B', 'device': 'auto', 'dtype': 'bfloat16', 'attn_implementation': 'sdpa', 'language': 'English'}
Model not loaded (lazy): OK


In [ ]:
# Test config change detection
print("Re-initializing with same config (no reload)...")
plugin.initialize({"language": "English", "device": "auto"})

print("Re-initializing with different dtype (would reload if model loaded)...")
plugin.initialize({"language": "English", "device": "auto", "dtype": "float16"})
print(f"Updated dtype: {plugin._config.dtype}")

# Cleanup
plugin.cleanup()
print("Cleanup: OK")

Initialized with model=Qwen/Qwen3-ForcedAligner-0.6B, device=auto
Initialized with model=Qwen/Qwen3-ForcedAligner-0.6B, device=auto
Cleanup completed


Re-initializing with same config (no reload)...
Re-initializing with different dtype (would reload if model loaded)...
Updated dtype: float16
Cleanup: OK


### Execution Tests

These tests require the Qwen3 model to be downloaded and GPU available.

In [ ]:
#| eval: false
from pathlib import Path

# Test execution with real audio
plugin = Qwen3ForcedAlignerPlugin()
plugin.initialize({"language": "English"})

test_audio = Path("../test_files/short_test_audio.mp3")
test_text = Path("../test_files/short_test_audio.txt").read_text().strip()

print(f"Audio: {test_audio}")
print(f"Text: {test_text[:80]}...")

result = plugin.execute(str(test_audio), text=test_text)

print(f"\nResult: {len(result.items)} words")
print(f"Metadata: {result.metadata}")
print(f"\nFirst 10 items:")
for item in result.items[:10]:
    print(f"  {item}")

plugin.cleanup()

Initialized with model=Qwen/Qwen3-ForcedAligner-0.6B, device=auto
Loading model Qwen/Qwen3-ForcedAligner-0.6B on cuda:0 with bfloat16


Audio: ../test_files/short_test_audio.mp3
Text: November the 10th, Wednesday, 9 p.m. I'm standing in a dark alley. After waiting...


Model loaded successfully
Running alignment on ../test_files/short_test_audio.mp3 (233 chars)
Saved result to DB (Job: a17c3e67-34f7-4f93-a38a-d53e0bc4c65c)
Alignment complete: 43 words
Unloading model
Cleanup completed



Result: 43 words
Metadata: {'model_id': 'Qwen/Qwen3-ForcedAligner-0.6B', 'language': 'English', 'device': 'cuda:0', 'dtype': 'bfloat16', 'word_count': 43, 'job_id': 'a17c3e67-34f7-4f93-a38a-d53e0bc4c65c'}

First 10 items:
  ForcedAlignItem(text='November', start_time=1.04, end_time=1.6)
  ForcedAlignItem(text='the', start_time=1.6, end_time=1.68)
  ForcedAlignItem(text='10th', start_time=1.76, end_time=2.08)
  ForcedAlignItem(text='Wednesday', start_time=2.48, end_time=3.04)
  ForcedAlignItem(text='9', start_time=3.84, end_time=4.16)
  ForcedAlignItem(text='pm', start_time=4.16, end_time=4.64)
  ForcedAlignItem(text="I'm", start_time=6.32, end_time=6.48)
  ForcedAlignItem(text='standing', start_time=6.48, end_time=7.2)
  ForcedAlignItem(text='in', start_time=7.2, end_time=7.28)
  ForcedAlignItem(text='a', start_time=7.28, end_time=7.36)


In [ ]:
#| eval: false
# Test with longer audio
plugin = Qwen3ForcedAlignerPlugin()
plugin.initialize({"language": "English"})

test_audio = Path("../test_files/02 - 1. Laying Plans.mp3")
test_text = Path("../test_files/02 - 1. Laying Plans.txt").read_text().strip()

print(f"Audio: {test_audio}")
print(f"Text length: {len(test_text)} chars, {len(test_text.split())} words")

result = plugin.execute(str(test_audio), text=test_text)

print(f"\nResult: {len(result.items)} words aligned")
print(f"First 5: {result.items[:5]}")
print(f"Last 5: {result.items[-5:]}")

# Verify storage
if plugin._storage:
    job_id = result.metadata.get("job_id")
    if job_id:
        row = plugin._storage.get_by_job_id(job_id)
        print(f"\nStored job: {row.job_id}")
        print(f"Stored items count: {len(row.items)}")
        assert row.items[0]["text"] == result.items[0].text
        print("Storage round-trip: OK")

plugin.cleanup()

Initialized with model=Qwen/Qwen3-ForcedAligner-0.6B, device=auto
Loading model Qwen/Qwen3-ForcedAligner-0.6B on cuda:0 with bfloat16


Audio: ../test_files/02 - 1. Laying Plans.mp3
Text length: 3405 chars, 599 words


Model loaded successfully
Running alignment on ../test_files/02 - 1. Laying Plans.mp3 (3405 chars)
Saved result to DB (Job: 66363e20-0fda-4994-93c4-2ecd328b6dcd)
Alignment complete: 599 words
Unloading model
Cleanup completed



Result: 599 words aligned
First 5: [ForcedAlignItem(text='Laying', start_time=1.12, end_time=1.52), ForcedAlignItem(text='Plans', start_time=1.52, end_time=2.24), ForcedAlignItem(text='Sun', start_time=4.8, end_time=5.04), ForcedAlignItem(text='Tzu', start_time=5.04, end_time=5.28), ForcedAlignItem(text='said', start_time=5.28, end_time=5.76)]
Last 5: [ForcedAlignItem(text='likely', start_time=250.88, end_time=251.36), ForcedAlignItem(text='to', start_time=251.36, end_time=251.36), ForcedAlignItem(text='win', start_time=251.44, end_time=252.0), ForcedAlignItem(text='or', start_time=252.32, end_time=252.4), ForcedAlignItem(text='lose', start_time=252.4, end_time=252.96)]

Stored job: 66363e20-0fda-4994-93c4-2ecd328b6dcd
Stored items count: 599
Storage round-trip: OK


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()